# 01 - Détection automatique du schéma

## Objectif

Ce notebook identifie automatiquement les rôles techniques des colonnes du dataset RetailSenseAI :

- variables numériques ;
- variables catégorielles ;
- variables booléennes ;
- variables datetime ;
- identifiants ;
- colonne cible.

Le notebook est autonome : si le CSV n'existe pas encore, il génère une version e-commerce déterministe avant l'analyse.

In [ ]:
# Imports et chemins
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.api.types import (
    is_bool_dtype,
    is_datetime64_any_dtype,
    is_numeric_dtype,
)

RANDOM_SEED = 42
CUSTOMER_COUNT = 1_200
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'notebooks' / 'data' / 'retailsense_ecommerce_customers.csv'
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

print('Dataset :', DATA_PATH)

## 1. Chargement autonome

La fonction suivante est un générateur de secours. Elle n'est appelée que si le CSV RetailSenseAI est absent. Ainsi, ce notebook fonctionne dans un kernel neuf et ne dépend pas de l'exécution du notebook `00`.

In [ ]:
def create_retailsense_dataset(path: Path, row_count: int = CUSTOMER_COUNT) -> None:
    """Crée le dataset e-commerce déterministe utilisé par la phase 1."""
    rng = np.random.default_rng(RANDOM_SEED)
    customer_id = [f'CUST-{index:06d}' for index in range(1, row_count + 1)]
    age = np.clip(rng.normal(39, 12, row_count).round(), 18, 80).astype(float)
    gender = rng.choice(['Femme', 'Homme', 'Non renseigné'], row_count, p=[0.49, 0.48, 0.03])
    country = rng.choice(['France', 'Belgique', 'Suisse'], row_count, p=[0.68, 0.20, 0.12])
    city_choices = {'France': ['Paris', 'Lyon', 'Lille', 'Bordeaux'], 'Belgique': ['Bruxelles', 'Liège', 'Namur'], 'Suisse': ['Genève', 'Lausanne', 'Zurich']}
    city = [rng.choice(city_choices[value]) for value in country]
    channel = rng.choice(['Organic', 'Paid Search', 'Social Media', 'Referral', 'Marketplace'], row_count)
    loyalty = rng.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], row_count, p=[0.44, 0.31, 0.19, 0.06])
    bonus = pd.Series(loyalty).map({'Bronze': 0, 'Silver': 2, 'Gold': 5, 'Platinum': 9}).to_numpy()
    frequency = np.maximum(1, rng.poisson(4 + bonus * 0.45, row_count)).astype(float)
    order_value = np.clip(rng.lognormal(4.25, 0.48, row_count), 15, 450)
    orders = np.maximum(1, (frequency * rng.uniform(1.5, 4.5, row_count)).round()).astype(int)
    spent = order_value * orders * rng.uniform(0.92, 1.08, row_count)
    discount = np.clip(rng.beta(2.2, 4.5, row_count), 0, 1)
    inactivity = np.clip(rng.exponential(38, row_count).round(), 0, 365).astype(int)
    returns = rng.binomial(orders, np.clip(0.03 + discount * 0.13, 0, 0.35))
    newsletter = rng.random(row_count) < 0.62
    signup = pd.Timestamp('2025-12-31') - pd.to_timedelta(rng.integers(30, 1_800, row_count), unit='D')
    churn_logit = -2 + 0.025 * inactivity - 0.13 * frequency - 0.00022 * spent + 0.9 * discount + 0.08 * returns - 0.42 * newsletter - 0.10 * bonus
    churn = rng.random(row_count) < (1 / (1 + np.exp(-churn_logit)))
    frame = pd.DataFrame({
        'customer_id': customer_id, 'age': age, 'gender': gender, 'city': city, 'country': country,
        'acquisition_channel': channel, 'loyalty_level': loyalty, 'purchase_frequency': frequency,
        'average_order_value': order_value.round(2), 'total_orders': orders, 'total_spent': spent.round(2),
        'discount_usage': discount.round(3), 'last_purchase_days': inactivity, 'number_returns': returns,
        'payment_method': rng.choice(['Carte', 'PayPal', 'Virement', 'Apple Pay'], row_count),
        'device': rng.choice(['Mobile', 'Desktop', 'Tablet'], row_count, p=[0.59, 0.34, 0.07]),
        'newsletter_subscribed': newsletter, 'signup_date': signup,
        'record_source': 'retailsense_demo', 'churn': churn,
    })
    for column, fraction in {'age': 0.025, 'city': 0.020, 'average_order_value': 0.018, 'payment_method': 0.015}.items():
        indexes = rng.choice(frame.index, int(row_count * fraction), replace=False)
        frame.loc[indexes, column] = np.nan
    frame.loc[rng.choice(frame.index, 6, replace=False), 'total_spent'] *= 8
    frame = pd.concat([frame, frame.sample(8, random_state=RANDOM_SEED)], ignore_index=True)
    frame.to_csv(path, index=False, date_format='%Y-%m-%d')

if not DATA_PATH.exists():
    create_retailsense_dataset(DATA_PATH)

data = pd.read_csv(DATA_PATH)
print(f'{len(data):,} lignes x {len(data.columns)} colonnes chargées')
display(data.head(3))

## 2. Stratégie de détection

L'ordre des tests est important :

1. **Target** : recherche d'un nom conventionnel (`target`, `label`, `churn`, etc.) ;
2. **Boolean** : testé avant numeric, car Python considère les booléens comme une forme d'entier ;
3. **Datetime** : type datetime natif ou texte convertible à au moins 90 % ;
4. **Identifier** : nom finissant par `_id` et cardinalité presque unique ;
5. **Numerical** : types numériques restants ;
6. **Categorical** : colonnes restantes.

La fonction retourne à la fois les groupes et un tableau explicatif par colonne.

In [ ]:
TARGET_NAMES = ('target', 'label', 'outcome', 'churn', 'is_churn')

def is_datetime_candidate(series: pd.Series, column_name: str) -> bool:
    """Reconnaît une date native ou une chaîne presque entièrement convertible."""
    if is_datetime64_any_dtype(series):
        return True
    name_suggests_date = any(token in column_name.lower() for token in ('date', 'time', 'timestamp'))
    if not name_suggests_date or is_numeric_dtype(series) or is_bool_dtype(series):
        return False
    converted = pd.to_datetime(series, errors='coerce')
    return bool(converted.notna().mean() >= 0.90)

def detect_schema(frame: pd.DataFrame) -> tuple[dict[str, tuple[str, ...]], pd.DataFrame]:
    target_candidates = [column for column in frame if column.lower() in TARGET_NAMES]
    if len(target_candidates) != 1:
        raise ValueError(f'Une cible unique est attendue, trouvé : {target_candidates}')
    target = target_candidates[0]
    groups: dict[str, list[str]] = {
        'target': [target], 'boolean': [], 'datetime': [],
        'identifier': [], 'numerical': [], 'categorical': [],
    }
    rows = []
    for column in frame.columns:
        series = frame[column]
        if column == target:
            role, reason = 'target', 'Nom reconnu comme cible métier'
        elif is_bool_dtype(series):
            role, reason = 'boolean', 'Type pandas bool'
        elif is_datetime_candidate(series, column):
            role, reason = 'datetime', 'Nom temporel et valeurs convertibles'
        elif column.lower().endswith('_id') and series.nunique(dropna=True) / len(series) >= 0.90:
            role, reason = 'identifier', 'Nom *_id et cardinalité quasi unique'
        elif is_numeric_dtype(series):
            role, reason = 'numerical', 'Type pandas numérique'
        else:
            role, reason = 'categorical', 'Valeurs textuelles ou discrètes'
        if column != target:
            groups[role].append(column)
        rows.append({
            'colonne': column, 'type_pandas': str(series.dtype), 'rôle_détecté': role,
            'raison': reason, 'valeurs_uniques': series.nunique(dropna=True),
            'valeurs_manquantes': int(series.isna().sum()),
        })
    immutable_groups = {name: tuple(columns) for name, columns in groups.items()}
    return immutable_groups, pd.DataFrame(rows)

schema, schema_table = detect_schema(data)
display(schema_table)

## 3. Résultat par famille

L'affichage suivant rend la décision lisible sans parcourir le tableau ligne par ligne. L'identifiant est séparé des variables catégorielles car il ne doit généralement pas entrer directement dans un modèle.

In [ ]:
for role in ('target', 'identifier', 'numerical', 'categorical', 'boolean', 'datetime'):
    columns = schema[role]
    print(f'\n{role.upper()} ({len(columns)})')
    for column in columns:
        print(f'  - {column}')

role_counts = schema_table['rôle_détecté'].value_counts().sort_values()
ax = role_counts.plot(kind='barh', figsize=(9, 4.5), color='#168aad')
ax.set_title('Nombre de colonnes par rôle détecté')
ax.set_xlabel('Nombre de colonnes')
ax.set_ylabel('Rôle')
for container in ax.containers:
    ax.bar_label(container, padding=3)
plt.tight_layout()
plt.show()

## 4. Validation automatique

Un schéma n'est utilisable que si chaque colonne reçoit exactement un rôle et si la cible est bien isolée des features. Ces assertions arrêtent immédiatement le notebook en cas d'ambiguïté.

In [ ]:
detected_columns = set().union(*schema.values())
assert detected_columns == set(data.columns), 'Toutes les colonnes doivent être classées'
assert sum(len(columns) for columns in schema.values()) == len(data.columns), 'Une colonne ne doit avoir qu’un rôle'
assert schema['target'] == ('churn',), 'La cible attendue est churn'
assert 'newsletter_subscribed' in schema['boolean']
assert 'signup_date' in schema['datetime']
assert 'customer_id' in schema['identifier']
assert 'churn' not in schema['numerical'] + schema['categorical'] + schema['boolean']

print('Validation réussie : schéma complet, exclusif et cible isolée.')

## Conclusion

Le schéma RetailSenseAI est détecté automatiquement et expliqué colonne par colonne. Les groupes obtenus pourront alimenter plus tard le nettoyage et le `ColumnTransformer`, mais ce notebook ne modifie aucune donnée.